# Homework 9 — solution by (your_name)

Read the homework PDF first: it has the rules. The questions are quoted in this notebook, in blockquotes (with a bar on
their left), so that your answers stand apart. Copy this notebook to
`3dasm_course/Assignments/your_Assignments/HW9/your_Homework9.ipynb` and work there.

**How this notebook works** (as in the lectures and in Homeworks 4 to 8):

1. the cell `%%writefile CLAUDE.md` writes the instructions file that the agent of this homework reads: **you** write it;
2. the prompt is a `PROMPT_...` cell, run with `run_agent`: **you** write it. The agent works in its own task folder
   inside `HW9/` (`HW9/ex2_2/`);
3. after the prompt, a cell that we provide loads the agent's `results.json`. The keys it needs are written just above
   the prompt cell: your prompt must make the agent store exactly those. If one is missing, the cell tells you which.

This is a **short** homework (two exercises), so that you have time for the Final Project.

**Your prompt is graded** (see the PDF): write it in your own words, and make it define the problem uniquely.
A prompt that only points to a question ("solve exercise 2.2"), or pastes it, counts as not answered.

In [ ]:
# Provided by the course: the helper that runs a coding agent from this notebook (see agent_helper.py)
import sys; sys.path.insert(0, "../../..")          # the folder 3dasm_course, where agent_helper.py is
from agent_helper import set_model, run_agent, results, figures, show, save_results, task_folder
import numpy as np, pandas as pd, scipy, torch
import matplotlib.pyplot as plt

set_model("sonnet")        # "sonnet" or "haiku": the TA runs your notebook again with this same model
import sklearn; print("numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__, "| torch", torch.__version__)   # some results depend on them

## Your instructions file for the agent

Write here everything that does not depend on the task: the environment, the packages, where files go, and what the
agent must report. The lectures and Homeworks 4 to 8 have examples.

In [ ]:
%%writefile CLAUDE.md
# YOUR CLAUDE.md HERE

## Exercise 1 (pen and paper)

> **1.1.** **A boundary that logistic regression cannot draw**: a QDA classifier with one input $x$ has estimated the class probabilities $\hat{\rho}_0 = \hat{\rho}_1 = 1/2$, the means $\hat{\mu}_0 = \hat{\mu}_1 = 0$ and the standard deviations $\hat{\sigma}_0 = 1$ and $\hat{\sigma}_1 = 2$. Find the values of $x$ for which the predicted class is $\hat{y}^* = 1$. Can logistic regression with $\boldsymbol{\phi}(x) = [1, x]^T$ predict these classes? Which basis functions $\boldsymbol{\phi}(x)$ would let it? What would LDA (one shared standard deviation) predict for these two classes?

*YOUR ANSWER HERE*

> **1.2.** **Gradient descent on a bowl**: let $f(\mathbf{x}) = \frac{1}{2}(\lambda_1 x_1^2 + \lambda_2 x_2^2)$ with $0 < \lambda_1 \leq \lambda_2$, and let $\kappa = \lambda_2/\lambda_1$. Show that one iteration of gradient descent with step size $\alpha$ multiplies each coordinate by $1 - \alpha\lambda_i$. For which $\alpha$ does gradient descent converge from every starting point? With $\alpha = 1/\lambda_2$, show that the number of iterations needed to reduce $x_1$ by a factor of 1000 is about $\kappa \ln 1000$ for large $\kappa$ (use $\ln(1-u) \approx -u$ for small $u$). How many iterations does Newton’s method need, and why?

*YOUR ANSWER HERE*

> **1.3.** **The step of a finite difference**: the forward difference $\left(f(x+h) - f(x)\right)/h$ estimates $f'(x)$. Use the Taylor series to show that its error is about $\frac{h}{2}|f''(x)|$. A computer stores $f$ with a relative error of about $\epsilon \approx 10^{-16}$ (double precision), so each of the two values of $f$ can be off by $\epsilon|f(x)|$: show that this adds an error of up to about $2\epsilon|f(x)|/h$. Find the $h$ that minimizes the sum of the two errors, and estimate it for $f(x) = e^x$ at $x = 1$. How many correct digits does the best forward difference give? How many evaluations of $f$ does a gradient by forward differences cost when $\mathbf{x}$ has $D_x$ components?

*YOUR ANSWER HERE*

## Exercise 2 (with the agent)

In this exercise you compare three optimizers on the Rosenbrock function in $D$ dimensions, $$f(\mathbf{x}) = \sum_{i=1}^{D-1}\left[100\left(x_{i+1} - x_i^2\right)^2 + \left(1 - x_i\right)^2\right],$$ whose global minimum is $f = 0$ at $\mathbf{x} = [1, \ldots, 1]$, for $D \in \{2, 8, 32\}$. For each $D$, the 10 starting points are the rows of `np.random.default_rng(123).uniform(-2, 2, size=(10, D))` (a new generator for each $D$). For the first two optimizers below, compute $f$ with `scipy.optimize.rosen` (a finite difference amplifies the last digits of $f$, so another way of writing the same sum changes the number of evaluations by a few percent: Exercise 1.3 at work). All three optimizers are those of `scipy.optimize.minimize`:

-   **Nelder-Mead**, with the options `maxfev=100000`, `maxiter=100000`, `xatol=1e-8` and `fatol=1e-12`;

-   **L-BFGS-B with finite differences**: no gradient given (SciPy estimates it), with the options `maxiter=100000` and `maxfun=1000000`;

-   **L-BFGS-B with automatic differentiation**: the function returns $f$ and its exact gradient, computed by PyTorch’s `autograd` in double precision (`jac=True`), with the same options.

**Count every evaluation**: wrap the function in a counter of its calls, so that the calls SciPy makes for the finite differences are counted too. For the third optimizer, one call returns $f$ and its gradient, and counts as one evaluation. A run **succeeds** if its final $f$ is below $10^{-6}$.

> **2.1.** **Before running anything**, predict: how does the number of evaluations of each optimizer grow with $D$? By roughly what factor should L-BFGS-B with finite differences need more evaluations than L-BFGS-B with automatic differentiation, for each $D$? (You will not lose points for wrong predictions. You will lose points for not writing them.)

*YOUR ANSWER HERE*

> **2.2.** Write a prompt that runs the 90 optimizations and stores, for each $D$, optimizer and starting point: the number of evaluations, the final $f$ and the final $\mathbf{x}$; and, for each $D$ and optimizer: the median number of evaluations and the number of successful runs. Plot the median number of evaluations against $D$ for the three optimizers (logarithmic vertical axis).

Your prompt must make the agent store in `results.json`: `D` (the list [2, 8, 32]), and `NM`, `FD` and `AD` (Nelder-Mead, L-BFGS-B with finite differences, L-BFGS-B with automatic differentiation), each a dictionary with the lists `evals`, `f_final` and `x_final` (one entry per $D$, each a list over the 10 starting points; `x_final` holds the final points), `median_evals` and `successes` (one value per $D$).

In [ ]:
TASK_2_2 = "ex2_2"          # the task name (the cells below use it)
PROMPT_2_2 = """YOUR PROMPT HERE"""
run_agent(TASK_2_2, PROMPT_2_2)

In [ ]:
# Provided: the results of task ex2_2
r = results(TASK_2_2, ["D", "NM", "FD", "AD"])
print(pd.DataFrame({f"{m}: {k}": r[m][k] for m in ("NM", "FD", "AD") for k in ("median_evals", "successes")},
                   index=[f"D={d}" for d in r["D"]]).T.to_string())
show(*figures(TASK_2_2), width=600)

*YOUR ANSWER HERE*

> **2.3.** **Check** (your own code, in a notebook cell, using only the stored results): for each $D$, divide the median number of evaluations of L-BFGS-B with finite differences by that of L-BFGS-B with automatic differentiation, and compare it with $D+1$. Explain the agreement with 1.3 in one sentence. Some runs did not succeed with either version of L-BFGS-B: take one of them, compute the gradient and the eigenvalues of the Hessian of $f$ at its final $\mathbf{x}$ (with `torch.autograd`), and say what kind of point it is (Lecture 27).

In [ ]:
# YOUR CHECK HERE (your own code: the stored results only)

*YOUR ANSWER HERE*

> **2.4.** **What can you conclude?** Compare your predictions of 2.1 with the results. Why does Nelder-Mead fail for $D = 32$, and why do the two versions of L-BFGS-B fail on the same starting points? In Task 4.2 of the Final Project you minimize the prediction of a Gaussian process over three design variables, under a constraint given by a classifier. Could you use exact gradients there? Would finite differences be expensive with three variables? Would your answer change if every evaluation were an Abaqus simulation? Use this to justify one of the two optimizers that you add in Task 4.2.

*YOUR ANSWER HERE*